In [ ]:
import os

from langchain_openai import ChatOpenAI


def get_model():
    """获取聊天模型实例"""
    return ChatOpenAI(
        model="Qwen/Qwen2.5-7B-Instruct",
        base_url="https://api.siliconflow.cn/v1",
        api_key=os.environ.get("SILICONFLOW_API_KEY"),
        temperature=0.2,
    )


llm = get_model()
llm

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001ED89849A90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001ED8984A660>, root_client=<openai.OpenAI object at 0x000001ED89370C20>, root_async_client=<openai.AsyncOpenAI object at 0x000001ED8984A3C0>, model_name='Qwen/Qwen2.5-7B-Instruct', temperature=0.2, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.siliconflow.cn/v1')

# Model Context Protocol
模型上下文协议（MCP）是一个 **标准化** 应用程序向LLMs提供 **工具** 和 **上下文** 的方式的开放协议。

在Langchain中Agent可以通过 `langchain-mcp-adapters` 来使用MCP服务器上定义的工具



## 快速入门

`langchain-mcp-adapters` 使得Agent能够使用一个或多个MCP服务器上定义的工具

> [!tip]
> MultiServerMCPClient 默认是无状态的。每次调用工具都会创建一个新的MCP ClientSession ，执行工具，然后清理




## 自定义MCP
使用fastmcp这个库

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("Math")


@mcp.tool()  # 通过装饰器定义为mcp的tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("Weather")


@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"


if __name__ == "__main__":
    mcp.run(transport="streamable-http")

In [ ]:
# 完整示例代码见 Advanced-Usage/mcp/run_agent.py
#
# 运行方式：
# 终端1: python Advanced-Usage/mcp/weather_server.py
# 终端2: python Advanced-Usage/mcp/run_agent.py

MultiServerMCPClient的第一个参数：`connections: dict[str, Connection]`



## Transports 传输方式
MCP支持多种不同的 C/S 通信传输机制；包括HTTP、stdio标准输入输出等

### HTTP
http 传输，也称为 streamable-http ； 使用HTTP请求进行客户端和服务器之间的通信

可以在链接配置中的 `headers` 字段包含自定义标头（比如说身份验证或追踪）

```python
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "http",
            "url": "http://localhost:8000/mcp",
            "headers": {
                "Authorization": "Bearer YOUR_TOKEN",
                "X-Custom-Header": "custom-value"
            },
        }
    }
)
tools = await client.get_tools()    # 异步
agent = create_agent("openai:gpt-4.1", tools)
response = await agent.ainvoke({"messages": "what is the weather in nyc?"})
```

### STDIO 标准输入输出
客户端以子进程的方式启动服务器，并且通过标准输入输出进行通信。最适合本地工具和简单配置

```python
from langchain_mcp_adapters.client import MultiServerMCPClient

# from langchain.agents import create_agent

client = MultiServerMCPClient(
    connections={
        "math": {
            "transport": "stdio",
            "command": ["python"],
            "args": ["./mcp/math_server.py"],
        }
    }
)

tools = client.get_tools()

print(tools)

for t in tools:
    print(t.name)
```